In [ ]:
import torch
from torch import nn
import torchvision
from torch.utils import data
from tqdm import tqdm
import numpy as np
from datetime import datetime
import json
import wandb
import os

from models import DINO_CLIP, TwoCropsTransform, GaussianBlur
from datasets import ImageNet, Places365, ArtPlaces, Transforms

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print(torch.cuda.get_device_name())

In [ ]:
LOG_RUN = True

CLIP_MODEL_NAME = "ViT-B/32"
DINO_MODEL_NAME = "dinov2_vitb14"
DATASET = "Places365"
N = None
match DATASET:
    case "ArtPlacesTimesN":
        N = 12
BATCH_SIZE = 64
EPOCHS = 50
CRITERION = "cross_entropy"
OPTIMIZER = "sgd"
SCHEDULER = "cosine"
match SCHEDULER:
    case "step":
        SCHEDULE = [15, 30]
    case "cosine":
        SCHEDULE = None

MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4
LR = 0.001

DIM = 512
K = 65536
M = 0.999
T = 0.07

In [ ]:
config = {
    "clip_model_name": CLIP_MODEL_NAME,
    "dino_model_name": DINO_MODEL_NAME,
    # "architecture": ARCHITECTURE,
    # "pretrained": PRETRAINED,
    "dataset": DATASET,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LR,
    # "hidden_units": HIDDEN_UNITS,
    # "ssim_loss": USE_SSIM_LOSS,
    # "ssim_loss_scale": SSIM_SCALE,
    # "perceptual_loss_architecture": PERCEPTUAL_LOSS_ARCHITECTURE,
    # "perceptual_loss_scale": SCALE,
    "optimizer": OPTIMIZER,
    "optimizer_momentum": MOMENTUM,
    "optimizer_weight_decay": WEIGHT_DECAY,
    "scheduler": SCHEDULER,
    "schedule": SCHEDULE,
    # "scheduler_modulo": SCHEDULE,
    # "scheduler_gamma": GAMMA,
    "dim": DIM,
    "k": K,
    "m": M,
    "t": T,
}

In [ ]:
if LOG_RUN:
    wandb.login()

In [ ]:
NUM_WORKERS = 8

transform = torchvision.transforms.Compose([
    torchvision.transforms.RandomResizedCrop(224, scale=(0.2, 1.0)),
    torchvision.transforms.RandomApply(
        [torchvision.transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)],
        p=0.8,  # not strengthened
    ),
    torchvision.transforms.RandomGrayscale(p=0.2),
    torchvision.transforms.RandomApply([GaussianBlur([0.1, 2.0])], p=0.5),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# transform = torchvision.transforms.Compose([
#     torchvision.transforms.RandomResizedCrop(224, scale=(0.2, 1.0)),
#     torchvision.transforms.RandomGrayscale(p=0.2),
#     torchvision.transforms.ColorJitter(0.4, 0.4, 0.4, 0.4),
#     torchvision.transforms.RandomHorizontalFlip(),
#     torchvision.transforms.ToTensor(),
#     torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

transform = TwoCropsTransform(transform)

match DATASET:
    case "ImageNet":
        dataset = ImageNet(root=r"C:\Users\mariu\Documents\Studium\Praktikum\ImageNet_Subset", transform=transform)
        dataset_val = ImageNet(root=r"C:\Users\mariu\Documents\Studium\Praktikum\ImageNet_Subset", split="val", transform=transform)
    case "Places365":
        dataset = Places365(root=r"C:\Users\mariu\Documents\Studium\Praktikum\Places365_Subset", transform=transform)
        dataset_val = Places365(root=r"C:\Users\mariu\Documents\Studium\Praktikum\Places365_Subset", transform=transform, split="val")
    case "ArtPlaces":
        dataset = ArtPlaces(root=r"C:\Users\mariu\Documents\Development\Datasets\ArtPlaces_13371280", transform=transform)
        dataset_val = ArtPlaces(root=r"C:\Users\mariu\Documents\Development\Datasets\ArtPlaces_13371280", transform=transform, split="val")
    # case "ArtPlacesTimesN":
    #     dataset = ArtPlacesTimesN(N, root=r"C:\Users\mariu\Documents\Development\Datasets\ArtPlaces_13371280", transform=transform)
    #     dataset_val = ArtPlaces(root=r"C:\Users\mariu\Documents\Development\Datasets\ArtPlaces_13371280", transform=transform, split="val")

data_loader = data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
data_loader_val = data.DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

### Modell

In [ ]:
model = DINO_CLIP(
    dino_model_name=DINO_MODEL_NAME,
    clip_model_name=CLIP_MODEL_NAME,
    dim = DIM,
    K = K,
    m = M,
    T = T,
)
# model = model.train()
model = model.to(device)

In [ ]:
# for name, param in model.named_parameters():
#     if param.requires_grad:
#         print(name)

In [ ]:
if LOG_RUN:
    wandb.init(
        # set the wandb project where this run will be logged
        project="dino_clip",
        dir=r"C:\Users\mariu\Desktop",

        # track hyperparameters and run metadata
        config=config
    )

In [ ]:
dest = os.path.join(r"C:\Users\mariu\Documents\Studium\Masterprojekt\Gewichte", "dino_clip", DINO_MODEL_NAME.lower() + "_" + CLIP_MODEL_NAME.lower().replace("/", "") + "_" + DATASET.lower() + "_" + datetime.today().strftime('%Y%m%d_%H%M%S'))
os.makedirs(dest)

with open(os.path.join(dest, "constants.json"), "w") as file:
    json.dump(config, file)

def save_model(epoch=0):
    torch.save(model.state_dict(), os.path.join(dest, "state_dict_" + str(epoch) + ".pt"))

In [ ]:
# Criterion

match CRITERION:
    case "cross_entropy":
        criterion = nn.CrossEntropyLoss().to(device)


# Optimizer

match OPTIMIZER:
    case "sgd":
        optimizer = torch.optim.SGD(
            filter(lambda p: p.requires_grad, model.parameters()), # model.parameters(),
            LR,
            momentum=MOMENTUM,
            weight_decay=WEIGHT_DECAY,
        )


# Scheduler

match SCHEDULER:
    case "step":
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.1)
    case "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
for epoch in range(EPOCHS):
    # Train

    losses = []
    
    tqdm_data_loader = tqdm(data_loader, unit="batch")
    tqdm_data_loader.set_description(f"Epoch {epoch+1}/{EPOCHS}")
    for i, (images, _) in enumerate(tqdm_data_loader):
        images[0] = images[0].to(device)
        images[1] = images[1].to(device)
    
        output, target = model(im_q=images[0], im_k=images[1])
        loss = criterion(output, target)

        if (i+1) % 10 == 0 and LOG_RUN:
            wandb.log({
                "loss": loss.item(),
                "learning_rate": scheduler.get_last_lr()[-1], 
                "epoch": epoch + 1,
                "batch": i + 1,
            })

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        postfix = {
            "Loss": loss.item(),
        }

        tqdm_data_loader.set_postfix(postfix)
    

    # Val

    val_losses = []

    model.eval()
    with torch.no_grad():
        tqdm_data_loader_val = tqdm(data_loader_val, unit="batch")
        tqdm_data_loader_val.set_description(f"Epoch {epoch+1}/{EPOCHS}")
        for i, (images, _) in enumerate(tqdm_data_loader_val):
            images[0] = images[0].to(device)
            images[1] = images[1].to(device)

            output, target = model(im_q=images[0], im_k=images[1])
            loss = criterion(output, target)

            val_losses.append(loss.item())
    model.train()

    if LOG_RUN:
        wandb.log({
            "loss_avg": np.mean(losses),
            "val_loss_avg": np.mean(val_losses),
        })

    if (epoch+1) % 20 == 0 or epoch + 1 == EPOCHS:
        save_model(epoch=epoch + 1)
    
    if SCHEDULE is None or epoch in SCHEDULE:
        scheduler.step()

In [ ]:
if LOG_RUN:
    wandb.finish()